<a href="https://colab.research.google.com/github/senasagdc/adversarial-attacks-dl-test/blob/main/notebooks/madry_reproduction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Reproducing Madry et al. (ICLR 2018) on MNIST

*Towards Deep Learning Models Resistant to Adversarial Attacks* — CMP4509 term project.

**How to run:** Runtime → Change runtime type → **GPU (T4)**, then *Run all*.
Every training run writes a checkpoint after each epoch and every evaluation writes a JSON file,
so if the session disconnects just run all cells again: finished work is skipped and
unfinished runs resume.

1. First run with `QUICK = True` (≈5 min) to check the pipeline.
2. Then set `QUICK = False` and run the real experiments (spread over several sessions if needed).

| Exp. | Question (paper section) | Output |
|---|---|---|
| E1 | Does PGD adversarial training give robustness at ε = 0.3? (Table 1) | clean / FGSM / PGD accuracy, 3 seeds |
| E2 | Does robustness need capacity? (Sec. 4, Fig. 4) | accuracy vs. network scale |
| E3 | Do PGD restarts reach similar loss values? (Sec. 3, Fig. 1) | loss histograms |
| E4 | Sanity checks against gradient masking + transfer attacks | ε-sweep, step-sweep, black-box |

In [36]:
# ---------------- settings ----------------
QUICK = False         # True: tiny smoke run to test the pipeline. False: real experiments.
USE_DRIVE = True      # Colab: store checkpoints/results on Google Drive (survives disconnects)
REPO_URL = "https://github.com/senasagdc/adversarial-attacks-dl-test.git"   # <- your repo

if QUICK:
    E1_SEEDS, E1_EPOCHS = [0], 1
    E2_SCALES, E2_MODES, E2_SEEDS, E2_EPOCHS = [1, 16], ["natural", "pgd"], [0], 1
    N_TEST_SUB = 500       # images for the expensive evaluations
    E3_RESTARTS = 100
else:
    E1_SEEDS, E1_EPOCHS = [0, 1, 2], 20
    E2_SCALES, E2_MODES, E2_SEEDS, E2_EPOCHS = [1, 2, 4, 8, 16], ["natural", "fgsm", "pgd"], [0, 1, 2], 5
    N_TEST_SUB = 1000
    E3_RESTARTS = 1000

BATCH_SIZE, LR = 50, 1e-4          # as in the paper
TRAIN_STEPS = 40                   # PGD-40 for training, step size 0.01, eps 0.3

In [37]:
import os, sys, subprocess, time
IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle/working")

# get madry.py (when the notebook is opened directly from GitHub the repo is not on disk)
if not os.path.exists("madry.py") and not os.path.exists("../madry.py"):
    if not os.path.exists("repo"):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "repo"], check=True)
    sys.path.insert(0, os.path.abspath("repo"))
else:
    sys.path.insert(0, os.path.abspath("..") if os.path.exists("../madry.py") else os.path.abspath("."))

if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive/madry_repro"
elif IN_KAGGLE:
    BASE = "/kaggle/working/madry_repro"
else:
    BASE = os.path.abspath("madry_repro")
OUT = os.path.join(BASE, "quick" if QUICK else "full")
CKPT, RES, FIG = (os.path.join(OUT, d) for d in ("checkpoints", "results", "figures"))
for d in (CKPT, RES, FIG):
    os.makedirs(d, exist_ok=True)

import torch
import madry as M
assert torch.cuda.is_available(), "Enable a GPU runtime (Runtime -> Change runtime type -> T4 GPU)"
DEVICE = "cuda"
torch.backends.cudnn.benchmark = True      # faster convolutions (tiny run-to-run nondeterminism)
print(torch.__version__, torch.cuda.get_device_name(0), "| output dir:", OUT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
2.14.1+cu130 Tesla T4 | output dir: /content/drive/MyDrive/madry_repro/full


In [38]:
train_data, test_data = M.load_mnist("data", DEVICE)   # local disk is faster than Drive
x_test, y_test = test_data
print("train", tuple(train_data[0].shape), "test", tuple(x_test.shape),
      "pixel range", float(x_test.min()), float(x_test.max()))
for s in (1, 2, 4, 8, 16):
    print(f"scale {s:2d}: {M.count_params(M.MadryCNN(s)):,} parameters")

train (60000, 1, 28, 28) test (10000, 1, 28, 28) pixel range 0.0 1.0
scale  1: 13,514 parameters
scale  2: 52,506 parameters
scale  4: 206,954 parameters
scale  8: 821,706 parameters
scale 16: 3,274,634 parameters


## 0. Sanity checks of the attack code
On a randomly initialised network: adversarial images must stay inside the ε-ball and inside [0, 1],
and PGD must increase the loss.

In [39]:
import torch.nn.functional as F
torch.manual_seed(0)
net = M.MadryCNN(16).to(DEVICE)
xb, yb = x_test[:256], y_test[:256]
for name, xa in [("fgsm", M.fgsm(net, xb, yb)), ("pgd", M.pgd(net, xb, yb, steps=20))]:
    linf = (xa - xb).abs().amax().item()
    assert linf <= M.EPS + 1e-6 and xa.min() >= 0 and xa.max() <= 1, name
    with torch.no_grad():
        print(f"{name}: max|delta|={linf:.4f}  loss clean={F.cross_entropy(net(xb), yb):.3f}"
              f"  loss adv={F.cross_entropy(net(xa), yb):.3f}")
assert all(p.grad is None for p in net.parameters()), "attack must not write parameter grads"
print("attack checks passed")

fgsm: max|delta|=0.3000  loss clean=2.309  loss adv=2.339
pgd: max|delta|=0.3000  loss clean=2.309  loss adv=2.387
attack checks passed


## Pilot: time per epoch
Trains PGD-40 adversarial training for one epoch with progress prints. Two things to check:
(1) seconds per epoch, to plan the budget; (2) training accuracy rises above ~11%
(a network stuck at ~11% predicts a single class — the failure seen in the earlier attempt).

In [40]:
RUN_PILOT = not QUICK
if RUN_PILOT and not os.path.exists(os.path.join(RES, "pilot.json")):
    t0 = time.time()
    _, hist = M.train("pgd", 16, 100, 1, os.path.join(CKPT, "pilot.pt"), train_data,
                      BATCH_SIZE, LR, TRAIN_STEPS, log_every=200, device=DEVICE)
    sec = time.time() - t0
    M.save_json(os.path.join(RES, "pilot.json"), {"seconds_per_epoch": sec, "history": hist})
if os.path.exists(os.path.join(RES, "pilot.json")):
    sec = M.load_json(os.path.join(RES, "pilot.json"))["seconds_per_epoch"]
    print(f"PGD-40 training: {sec/60:.1f} min/epoch -> one E1 run of {E1_EPOCHS} epochs "
          f"≈ {sec*E1_EPOCHS/60:.0f} min; all {len(E1_SEEDS)} PGD runs ≈ {sec*E1_EPOCHS*len(E1_SEEDS)/3600:.1f} h")

    it 200: train loss 2.309, train acc (on pgd inputs) 0.094
    it 400: train loss 2.260, train acc (on pgd inputs) 0.158
    it 600: train loss 2.165, train acc (on pgd inputs) 0.214
    it 800: train loss 2.062, train acc (on pgd inputs) 0.256
    it 1000: train loss 1.973, train acc (on pgd inputs) 0.288
    it 1200: train loss 1.894, train acc (on pgd inputs) 0.316
  [pgd s=16 seed=100] epoch 1/1 loss 1.8940 acc 0.3161 (74s)
PGD-40 training: 1.2 min/epoch -> one E1 run of 20 epochs ≈ 25 min; all 3 PGD runs ≈ 1.2 h


## E1 — Natural vs. PGD adversarial training (paper Table 1)
Full paper architecture (scale 16), Adam lr 1e-4, batch 50, PGD-40 (α = 0.01, ε = 0.3, random start).
We use the **final** model (no checkpoint selection), as the paper does, so the test set is never used for choices.

In [ ]:
for seed in E1_SEEDS:
    for mode in ("natural", "pgd"):
        ck = os.path.join(CKPT, f"e1_{mode}_seed{seed}.pt")
        M.train(mode, 16, seed, E1_EPOCHS, ck, train_data, BATCH_SIZE, LR, TRAIN_STEPS, device=DEVICE)

  [natural s=16 seed=0] epoch 1/20 loss 0.2826 acc 0.9198 (4s)
  [natural s=16 seed=0] epoch 2/20 loss 0.0769 acc 0.9766 (4s)
  [natural s=16 seed=0] epoch 3/20 loss 0.0530 acc 0.9839 (4s)
  [natural s=16 seed=0] epoch 4/20 loss 0.0407 acc 0.9876 (4s)
  [natural s=16 seed=0] epoch 5/20 loss 0.0332 acc 0.9896 (4s)
  [natural s=16 seed=0] epoch 6/20 loss 0.0271 acc 0.9918 (4s)
  [natural s=16 seed=0] epoch 7/20 loss 0.0223 acc 0.9931 (4s)
  [natural s=16 seed=0] epoch 8/20 loss 0.0192 acc 0.9940 (4s)
  [natural s=16 seed=0] epoch 9/20 loss 0.0157 acc 0.9953 (4s)
  [natural s=16 seed=0] epoch 10/20 loss 0.0129 acc 0.9959 (4s)
  [natural s=16 seed=0] epoch 11/20 loss 0.0109 acc 0.9967 (4s)
  [natural s=16 seed=0] epoch 12/20 loss 0.0095 acc 0.9972 (4s)
  [natural s=16 seed=0] epoch 13/20 loss 0.0083 acc 0.9976 (4s)
  [natural s=16 seed=0] epoch 14/20 loss 0.0070 acc 0.9978 (4s)
  [natural s=16 seed=0] epoch 15/20 loss 0.0059 acc 0.9982 (4s)
  [natural s=16 seed=0] epoch 16/20 loss 0.0054 a

In [ ]:
# Evaluation. Full test set for single-start attacks; first N_TEST_SUB images for 20 restarts.
xs, ys = x_test[:N_TEST_SUB], y_test[:N_TEST_SUB]
for seed in E1_SEEDS:
    for mode in ("natural", "pgd"):
        out = os.path.join(RES, f"e1_{mode}_seed{seed}.json")
        if os.path.exists(out):
            continue
        model = M.load_model(os.path.join(CKPT, f"e1_{mode}_seed{seed}.pt"), DEVICE)
        t0 = time.time()
        r = {"mode": mode, "seed": seed,
             "clean": M.evaluate(model, x_test, y_test, "none"),
             "fgsm": M.evaluate(model, x_test, y_test, "fgsm"),
             "pgd40": M.evaluate(model, x_test, y_test, "pgd", steps=40),
             "pgd100": M.evaluate(model, x_test, y_test, "pgd", steps=100),
             f"sub{N_TEST_SUB}_pgd40x1": M.evaluate(model, xs, ys, "pgd", steps=40, restarts=1),
             f"sub{N_TEST_SUB}_pgd40x20": M.evaluate(model, xs, ys, "pgd", steps=40, restarts=20),
             f"sub{N_TEST_SUB}_pgd100x20": M.evaluate(model, xs, ys, "pgd", steps=100, restarts=20)}
        r["eval_seconds"] = time.time() - t0
        M.save_json(out, r)
        print(r)

## E2 — Capacity (paper Sec. 4, Fig. 4)
Scale s gives a network with 2s / 4s filters and 64s hidden units (s = 16 is the E1 network).
Shorter budget (E2_EPOCHS) to fit free GPUs.

In [ ]:
x_tr_sub, y_tr_sub = train_data[0][:N_TEST_SUB], train_data[1][:N_TEST_SUB]
for scale in E2_SCALES:
    for mode in E2_MODES:
        for seed in E2_SEEDS:
            tag = f"e2_{mode}_scale{scale}_seed{seed}"
            out = os.path.join(RES, tag + ".json")
            if os.path.exists(out):
                continue
            model, hist = M.train(mode, scale, seed, E2_EPOCHS, os.path.join(CKPT, tag + ".pt"),
                                  train_data, BATCH_SIZE, LR, TRAIN_STEPS, device=DEVICE)
            model.eval()
            r = {"mode": mode, "scale": scale, "seed": seed, "params": M.count_params(model),
                 "clean": M.evaluate(model, x_test, y_test, "none"),
                 "fgsm": M.evaluate(model, x_test, y_test, "fgsm"),
                 "pgd40": M.evaluate(model, x_test, y_test, "pgd", steps=40),
                 "train_clean": M.evaluate(model, x_tr_sub, y_tr_sub, "none"),
                 "train_pgd40": M.evaluate(model, x_tr_sub, y_tr_sub, "pgd", steps=40),
                 "final_train_loss": hist[-1]["train_loss"]}
            M.save_json(out, r)
            print(r)

## E3 — Loss values across PGD restarts (paper Fig. 1)
For a few test images, many PGD runs from random starts. The paper's claim: the final losses
are tightly concentrated (no much better local maxima), and much lower for the robust model.

In [ ]:
out = os.path.join(RES, "e3_loss_concentration.json")
if not os.path.exists(out):
    nat = M.load_model(os.path.join(CKPT, f"e1_natural_seed{E1_SEEDS[0]}.pt"), DEVICE)
    rob = M.load_model(os.path.join(CKPT, f"e1_pgd_seed{E1_SEEDS[0]}.pt"), DEVICE)
    with torch.no_grad():   # images correctly classified by both models
        ok = (nat(x_test[:200]).argmax(1) == y_test[:200]) & (rob(x_test[:200]).argmax(1) == y_test[:200])
    idx = ok.nonzero().flatten()[:5].tolist()
    torch.manual_seed(0)
    res = {"images": idx, "restarts": E3_RESTARTS}
    for name, model in (("natural", nat), ("pgd", rob)):
        res[name] = []
        for i in idx:
            c = M.pgd_loss_curves(model, x_test[i], y_test[i], restarts=E3_RESTARTS, steps=40)
            res[name].append({"final": c[:, -1].tolist(), "curves20": c[:20].tolist()})
    M.save_json(out, res)
print("E3 done")

## E4 — Sanity checks and transfer attacks
* **ε-sweep:** a truly robust model must still go to ~0% accuracy for large ε (otherwise gradients are masked).
* **Step-sweep:** more PGD steps must never increase accuracy.
* **Transfer (black-box):** PGD-100 examples crafted on an independently trained natural model (paper's source A′).

In [ ]:
out = os.path.join(RES, "e4_sanity.json")
if not os.path.exists(out):
    EPS_GRID = [0.0, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.5]
    STEP_GRID = [1, 5, 10, 20, 40, 100, 200]
    res = {"eps_grid": EPS_GRID, "step_grid": STEP_GRID, "runs": []}
    for seed in E1_SEEDS:
        for mode in ("natural", "pgd"):
            model = M.load_model(os.path.join(CKPT, f"e1_{mode}_seed{seed}.pt"), DEVICE)
            # step size scaled with eps so that 100 steps can reach the boundary
            eps_acc = [M.evaluate(model, xs, ys, "pgd", eps=e, alpha=max(e / 30, 1e-4), steps=100) if e > 0
                       else M.evaluate(model, xs, ys, "none") for e in EPS_GRID]
            step_acc = [M.evaluate(model, xs, ys, "pgd", steps=k) for k in STEP_GRID]
            src = M.load_model(os.path.join(CKPT, f"e1_natural_seed{E1_SEEDS[(E1_SEEDS.index(seed) + 1) % len(E1_SEEDS)]}.pt"), DEVICE)
            transfer = M.transfer_accuracy(src, model, x_test, y_test, steps=100)
            res["runs"].append({"mode": mode, "seed": seed, "eps_acc": eps_acc,
                                "step_acc": step_acc, "transfer_pgd100": transfer})
            print(mode, seed, "transfer:", transfer)
    M.save_json(out, res)
print("E4 done")

## Summary tables and figures (used in the report)
Mean ± sample standard deviation (ddof = 1) over seeds.

In [ ]:
import glob, json
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 120, "font.size": 10})

def ms(v):
    v = np.asarray(v, float) * 100
    return f"{v.mean():.2f} ± {v.std(ddof=1):.2f}" if len(v) > 1 else f"{v.mean():.2f}"

# ---- E1 table ----
e1 = pd.DataFrame([M.load_json(p) for p in sorted(glob.glob(os.path.join(RES, "e1_*_seed*.json")))])
cols = [c for c in e1.columns if c not in ("mode", "seed", "eval_seconds")]
table1 = e1.groupby("mode")[cols].agg(ms).T
table1.to_csv(os.path.join(RES, "table_e1.csv"))
display(table1)
display(e1.set_index(["mode", "seed"])[cols].mul(100).round(2))   # per-seed values

# ---- Figure: E1 bars ----
fig, ax = plt.subplots(figsize=(6, 3.2))
metrics = ["clean", "fgsm", "pgd40", "pgd100"]
for k, mode in enumerate(("natural", "pgd")):
    d = e1[e1["mode"] == mode][metrics] * 100
    ax.bar(np.arange(4) + (k - 0.5) * 0.38, d.mean(), 0.38, yerr=d.std(ddof=1) if len(d) > 1 else None,
           capsize=3, label={"natural": "natural training", "pgd": "PGD adversarial training"}[mode])
    for j, m in enumerate(metrics):     # individual seeds as dots
        ax.scatter(np.full(len(d), j + (k - 0.5) * 0.38), d[m], color="k", s=8, zorder=3)
ax.set_xticks(range(4), ["clean", "FGSM", "PGD-40", "PGD-100"])
ax.set_ylabel("test accuracy (%)"); ax.set_ylim(0, 100); ax.legend(loc="lower left")
ax.set_title("MNIST, ε = 0.3")
fig.tight_layout(); fig.savefig(os.path.join(FIG, "e1_accuracy.pdf")); fig.savefig(os.path.join(FIG, "e1_accuracy.png"))

# ---- Figure: E1 training curves ----
fig, ax = plt.subplots(figsize=(6, 3))
for seed in E1_SEEDS:
    for mode, c in (("natural", "C0"), ("pgd", "C1")):
        h = torch.load(os.path.join(CKPT, f"e1_{mode}_seed{seed}.pt"), map_location="cpu", weights_only=False)["history"]
        ax.plot([r["epoch"] for r in h], [r["train_loss"] for r in h], color=c, alpha=0.7,
                label=mode if seed == E1_SEEDS[0] else None)
ax.set_xlabel("epoch"); ax.set_ylabel("training loss (on training inputs)"); ax.set_yscale("log"); ax.legend()
fig.tight_layout(); fig.savefig(os.path.join(FIG, "e1_training_loss.pdf")); fig.savefig(os.path.join(FIG, "e1_training_loss.png"))

In [ ]:
# ---- E2: capacity ----
e2 = pd.DataFrame([M.load_json(p) for p in glob.glob(os.path.join(RES, "e2_*.json"))])
e2.to_csv(os.path.join(RES, "e2_all_runs.csv"), index=False)
display(e2.groupby(["mode", "scale"])[["clean", "fgsm", "pgd40", "train_pgd40"]].agg(ms))
modes = [m for m in ("natural", "fgsm", "pgd") if m in set(e2["mode"])]
fig, axes = plt.subplots(1, len(modes), figsize=(3.3 * len(modes), 3), sharey=True, squeeze=False)
for ax, mode in zip(axes[0], modes):
    for metric, lab in (("clean", "clean"), ("fgsm", "FGSM"), ("pgd40", "PGD-40")):
        g = e2[e2["mode"] == mode].groupby("scale")[metric]
        m, s = g.mean() * 100, g.std(ddof=1).fillna(0) * 100
        ax.errorbar(m.index, m, yerr=s, marker="o", capsize=3, label=lab)
    ax.set_xscale("log", base=2); ax.set_xticks(E2_SCALES, [str(s) for s in E2_SCALES])
    ax.set_title({"natural": "natural training", "fgsm": "FGSM training", "pgd": "PGD training"}[mode])
    ax.set_xlabel("capacity scale"); ax.set_ylim(0, 100)
axes[0, 0].set_ylabel("test accuracy (%)"); axes[0, -1].legend()
fig.tight_layout(); fig.savefig(os.path.join(FIG, "e2_capacity.pdf")); fig.savefig(os.path.join(FIG, "e2_capacity.png"))

In [ ]:
# ---- E3: loss concentration ----
e3 = M.load_json(os.path.join(RES, "e3_loss_concentration.json"))
fig, axes = plt.subplots(2, len(e3["images"]), figsize=(2.4 * len(e3["images"]), 4.4), squeeze=False)
for j in range(len(e3["images"])):
    for i, (name, c) in enumerate((("natural", "C0"), ("pgd", "C1"))):
        axes[i, j].hist(e3[name][j]["final"], bins=30, color=c)
        axes[i, j].set_title(f"{name}, img {e3['images'][j]}", fontsize=8)
        axes[i, j].tick_params(labelsize=7)
for i in range(2):
    axes[i, 0].set_ylabel("count")
fig.supxlabel(f"cross-entropy loss after PGD-40 ({e3['restarts']} random restarts)", fontsize=9)
fig.tight_layout(); fig.savefig(os.path.join(FIG, "e3_loss_hist.pdf")); fig.savefig(os.path.join(FIG, "e3_loss_hist.png"))

fig, axes = plt.subplots(1, 2, figsize=(7, 2.8))
for ax, (name, c) in zip(axes, (("natural", "C0"), ("pgd", "C1"))):
    for curve in e3[name][0]["curves20"]:
        ax.plot(curve, color=c, alpha=0.5, lw=0.8)
    ax.set_title(f"{name} training, image {e3['images'][0]}"); ax.set_xlabel("PGD iteration")
axes[0].set_ylabel("loss"); fig.tight_layout()
fig.savefig(os.path.join(FIG, "e3_loss_curves.pdf")); fig.savefig(os.path.join(FIG, "e3_loss_curves.png"))
for name in ("natural", "pgd"):
    f = np.array([d["final"] for d in e3[name]])
    print(f"{name}: final loss mean {f.mean():.3f}, mean within-image std {f.std(1).mean():.4f}, "
          f"max {f.max():.3f}")

In [ ]:
# ---- E4: sanity checks ----
e4 = M.load_json(os.path.join(RES, "e4_sanity.json"))
fig, axes = plt.subplots(1, 2, figsize=(7.5, 3))
for mode, c in (("natural", "C0"), ("pgd", "C1")):
    runs = [r for r in e4["runs"] if r["mode"] == mode]
    for key, ax, grid in (("eps_acc", axes[0], e4["eps_grid"]), ("step_acc", axes[1], e4["step_grid"])):
        a = np.array([r[key] for r in runs]) * 100
        ax.errorbar(grid, a.mean(0), yerr=a.std(0, ddof=1) if len(a) > 1 else None, marker="o",
                    capsize=3, color=c, label=mode)
axes[0].axvline(0.3, ls="--", color="gray"); axes[0].set_xlabel("ε (PGD-100)"); axes[0].set_ylabel("accuracy (%)")
axes[1].set_xscale("log"); axes[1].set_xlabel("PGD steps (ε = 0.3)"); axes[0].legend()
fig.tight_layout(); fig.savefig(os.path.join(FIG, "e4_sanity.pdf")); fig.savefig(os.path.join(FIG, "e4_sanity.png"))
for mode in ("natural", "pgd"):
    print(f"transfer PGD-100 from an independent natural model -> {mode}:",
          ms([r["transfer_pgd100"] for r in e4["runs"] if r["mode"] == mode]))
print("figures saved to", FIG)